# Assertions — Signal Validation and Article Diagnostics

This notebook independently audits `notebooks/00_R_Signal.ipynb`. It does **not** publish signals, mutate source data, execute another notebook, or overwrite article artifacts.

The audit separates two classes of checks:

- **hard contracts** — schema, point-in-time alignment, complete windows, cache identity, numerical finiteness, and reproducibility; any violation raises an `AssertionError`;
- **scientific diagnostics** — correlations, power, AUC, persistence, and explanatory shares; these are reported transparently and are not converted into arbitrary pass/fail claims.

The volatility-spanning block is recomputed from the Parquet inputs. Its stored numerical expectations are regression guards for the frozen article sample (1995-01 to 2025-12), not hypothesis tests.

## 1. Setup and audit policy

Paths are resolved from either the repository root or the `tests/` directory. All source files are opened read-only. The only side effect of running this notebook is its own displayed output.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import polars as pl
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score
from statsmodels.tsa.stattools import adfuller

ROOT = Path.cwd()
if ROOT.name in {'tests', 'notebooks'}:
    ROOT = ROOT.parent

DATA   = ROOT / 'data'
SIG    = DATA / 'signals'
PANEL  = DATA / 'processed' / 'nyse_big_caps_pit_daily.parquet'
VIX    = DATA / 'processed' / 'vix_1990.parquet'
OUT    = ROOT / 'outputs' / 'results' / 'signal'
CACHE  = ROOT / 'cache' / 'signal'   # 03's working files live under the project cache root
IMG    = OUT / 'images'
TAB    = OUT / 'tables'
NB_ART = ROOT / 'notebooks' / '03_R_Signal.ipynb'

ARTICLE_START = pd.Period('1995-01', freq='M')
ARTICLE_END   = pd.Period('2025-12', freq='M')
ARTICLE_MONTHS = pd.period_range(ARTICLE_START, ARTICLE_END, freq='M')
EXPECTED_N_MONTHS = len(ARTICLE_MONTHS)  # 372
HAC_LAGS = 36

CHECKS = []

def require(name, condition, detail=''):
    ok = bool(condition)
    if not ok:
        raise AssertionError(f'{name} FAILED' + (f' | {detail}' if detail else ''))
    CHECKS.append(name)
    print(f'PASS  {name}' + (f' | {detail}' if detail else ''))

def finite(values):
    return np.isfinite(np.asarray(values, dtype=float)).all()

for path in [SIG, PANEL, VIX, OUT, NB_ART]:
    require(f'path exists: {path.relative_to(ROOT)}', path.exists())

print(f'Article audit window: {ARTICLE_START} -> {ARTICLE_END} ({EXPECTED_N_MONTHS} months)')

## 2. Published signal contracts

Every reference and robustness series must have the same raw monthly support, a unique month key, finite non-negative values, and complete coverage of the 372-month article sample. The December 1994 warm-up observation may remain in the published files but is excluded from article diagnostics.

In [ ]:
SIGNAL_FILES = [
    'V_uni.parquet', 'V_small_uni.parquet',
    'U_M36.parquet', 'U_M72.parquet',
    'U_lamTk.parquet', 'U_lamlog.parquet',
    'U_volscale.parquet', 'U_H04.parquet', 'U_H06.parquet',
    'U_exact.parquet', 'U_bary1d.parquet',
]

signals = {}
article_signals = {}
raw_periods = None

for fname in SIGNAL_FILES:
    path = SIG / fname
    require(f'signal exists: {fname}', path.exists())
    frame = pd.read_parquet(path)
    require(f'{fname}: schema', {'date', 'rho'} <= set(frame.columns))
    frame = frame[['date', 'rho']].copy()
    frame['date'] = pd.to_datetime(frame['date'])
    frame['period'] = frame['date'].dt.to_period('M')
    require(f'{fname}: unique dates', frame['date'].is_unique)
    require(f'{fname}: unique months', frame['period'].is_unique)
    require(f'{fname}: ordered dates', frame['date'].is_monotonic_increasing)
    require(f'{fname}: finite rho', finite(frame['rho']))
    require(f'{fname}: non-negative rho', frame['rho'].ge(0).all())

    rp = pd.PeriodIndex(frame['period'])
    if raw_periods is None:
        raw_periods = rp
    else:
        require(f'{fname}: common raw month support', rp.equals(raw_periods))

    article = frame.loc[frame['period'].between(ARTICLE_START, ARTICLE_END)].copy()
    require(f'{fname}: complete article sample',
            pd.PeriodIndex(article['period']).equals(ARTICLE_MONTHS),
            f'n={len(article)}')
    signals[fname] = frame
    article_signals[fname] = article.set_index('period')['rho']

require('reference raw support includes one warm-up month',
        len(signals['V_uni.parquet']) == EXPECTED_N_MONTHS + 1,
        f"raw n={len(signals['V_uni.parquet'])}")

manifest_path = SIG / 'rho_pit_validation_manifest.parquet'
require('validation manifest exists', manifest_path.exists())
manifest = pd.read_parquet(manifest_path)
require('validation manifest passed', manifest['validation_passed'].fillna(False).all())
require('validation manifest unique keys',
        ~manifest.duplicated(['universe', 'config_id']).any())

print(f'\nPublished series checked: {len(SIGNAL_FILES)}')

## 3. PIT universe and formation-membership contracts

The daily panel must have a unique security-date key. Formation membership must contain exactly 100 ranked names per month. The active daily universe may contain 98–100 names around data exits, but the number of valid daily returns must never fall below 98.

In [ ]:
REQUIRED_PANEL = {
    'PERMNO', 'DlyCalDt', 'DlyRet', 'is_active',
    'formation_member_month', 'formation_member_rank',
    'is_formation_member', 'formation_member_strict_hist60',
}
panel_lf = pl.scan_parquet(PANEL)
panel_cols = set(panel_lf.collect_schema().names())
require('PIT panel schema', REQUIRED_PANEL <= panel_cols,
        f'missing={sorted(REQUIRED_PANEL - panel_cols)}')

key_audit = panel_lf.select([
    pl.len().alias('rows'),
    pl.struct(['PERMNO', 'DlyCalDt']).n_unique().alias('unique_keys'),
    pl.col('DlyCalDt').min().alias('date_min'),
    pl.col('DlyCalDt').max().alias('date_max'),
]).collect(engine='streaming').row(0, named=True)
require('PIT panel primary key', key_audit['rows'] == key_audit['unique_keys'],
        f"rows={key_audit['rows']:,}")

members = (
    panel_lf.filter(pl.col('is_formation_member') == 1)
    .select([
        pl.col('formation_member_month').alias('formation_month'),
        'PERMNO',
        pl.col('formation_member_rank').alias('rank'),
        pl.col('formation_member_strict_hist60').alias('strict_hist60'),
        pl.col('DlyCalDt').alias('formation_date'),
    ])
    .collect(engine='streaming')
    .to_pandas()
)
members['formation_month'] = pd.to_datetime(members['formation_month'])
members['formation_date'] = pd.to_datetime(members['formation_date'])

member_audit = members.groupby('formation_month').agg(
    n=('PERMNO', 'size'), n_permno=('PERMNO', 'nunique'),
    rank_min=('rank', 'min'), rank_max=('rank', 'max'),
    n_rank=('rank', 'nunique'), n_dates=('formation_date', 'nunique'),
    strict60=('strict_hist60', 'all'),
)
require('formation membership cardinality',
        member_audit['n'].eq(100).all() & member_audit['n_permno'].eq(100).all())
require('formation rank contract',
        member_audit['rank_min'].eq(1).all()
        & member_audit['rank_max'].eq(100).all()
        & member_audit['n_rank'].eq(100).all())
require('one formation date per month', member_audit['n_dates'].eq(1).all())
require('strict 60-month eligibility', member_audit['strict60'].all())

active_daily = (
    panel_lf.filter(pl.col('is_active') == 1)
    .group_by('DlyCalDt')
    .agg([
        pl.len().alias('n_active'),
        pl.col('DlyRet').is_not_null().sum().alias('n_valid'),
    ])
    .collect(engine='streaming')
    .to_pandas()
)
require('active universe upper bound', active_daily['n_active'].max() == 100)
require('active universe lower bound', active_daily['n_active'].min() >= 98,
        f"min={active_daily['n_active'].min()}")
require('valid active returns lower bound', active_daily['n_valid'].min() >= 98,
        f"min={active_daily['n_valid'].min()}")

print(f"Formation months: {len(member_audit)} | active trading days: {len(active_daily):,}")

## 4. Independent 36-month window reconstruction

A wide daily-return matrix is reconstructed independently from the panel. Every article month must select the exact ranked formation membership, stop on the formation close, contain 100 columns, and be completely free of missing returns. Anchor windows are printed for manual inspection.

In [ ]:
daily_long = (
    pl.scan_parquet(PANEL)
    .select(['PERMNO', 'DlyCalDt', 'DlyRet'])
    .collect(engine='streaming')
    .to_pandas()
)
daily_long['DlyCalDt'] = pd.to_datetime(daily_long['DlyCalDt'])
require('daily long key remains unique',
        ~daily_long.duplicated(['PERMNO', 'DlyCalDt']).any())

pivot = (daily_long.pivot(index='DlyCalDt', columns='PERMNO', values='DlyRet')
         .sort_index())
pivot.columns = pivot.columns.astype(int)

memberships = {}
formation_dates = {}
for fm, group in members.groupby('formation_month', sort=True):
    ordered = group.sort_values('rank')
    memberships[pd.Timestamp(fm)] = ordered['PERMNO'].astype(int).tolist()
    formation_dates[pd.Timestamp(fm)] = pd.Timestamp(ordered['formation_date'].iloc[0])

article_fms = [pd.Timestamp(p.start_time) for p in ARTICLE_MONTHS]
require('all article formations available', set(article_fms) <= set(memberships))

def formation_window(fm):
    fm = pd.Timestamp(fm)
    assets = memberships[fm]
    formation_date = formation_dates[fm]
    start = fm - pd.DateOffset(months=35)
    frame = pivot.loc[(pivot.index >= start) & (pivot.index <= formation_date), assets]
    assert len(assets) == 100 and frame.shape[1] == 100, f'{fm:%Y-%m}: cardinality failure'
    assert frame.columns.tolist() == assets, f'{fm:%Y-%m}: rank/order mismatch'
    assert not frame.empty, f'{fm:%Y-%m}: empty window'
    assert frame.index.max() == formation_date, f'{fm:%Y-%m}: wrong window end'
    assert frame.index.min().to_period('M') == start.to_period('M'), f'{fm:%Y-%m}: incomplete start month'
    assert frame.notna().all().all(), f'{fm:%Y-%m}: missing return in strict PIT window'
    return frame

for anchor in ['1995-01', '2009-09', '2020-03', '2025-12']:
    frame = formation_window(pd.Timestamp(anchor + '-01'))
    print(f'{anchor}: {frame.index.min().date()} -> {frame.index.max().date()} '
          f'| days={len(frame)} | assets={frame.shape[1]}')

require('anchor PIT windows reconstructed', True)

## 5. Placebo, power, ROC, and Monte-Carlo cache contracts

These checks validate cache schemas, primary keys, grids, finiteness, and paired-window identities. Directional placebo separation, monotone injected-shock power, AUCs, and Monte-Carlo convergence are reported as scientific diagnostics.

In [ ]:
PLAC_PATH = CACHE / 'placebo' / 'placebo_h0h1.parquet'
POW_PATH  = CACHE / 'power' / 'power_curves.parquet'
ROC_PATH  = CACHE / 'power' / 'roc_intensity.parquet'
MC_PATH   = CACHE / 'monte_carlo' / 'mc_convergence.parquet'
for p in [PLAC_PATH, POW_PATH, ROC_PATH, MC_PATH]:
    require(f'cache exists: {p.relative_to(ROOT)}', p.exists())

placebo = pd.read_parquet(PLAC_PATH)
require('placebo schema', set(placebo.columns) == {'formation_month', 'h0', 'h1'})
require('placebo paired key', len(placebo) == 300 and placebo['formation_month'].is_unique)
require('placebo finite and non-negative',
        finite(placebo[['h0', 'h1']]) and placebo[['h0', 'h1']].ge(0).all().all())
sep_rate = float((placebo['h1'] > placebo['h0']).mean())
placebo_auc = roc_auc_score(
    np.r_[np.zeros(len(placebo)), np.ones(len(placebo))],
    np.r_[placebo['h0'], placebo['h1']],
)
print(f'Placebo diagnostics: P(H1>H0)={sep_rate:.3f} | AUC={placebo_auc:.3f}')

power = pd.read_parquet(POW_PATH)
require('power schema', set(power.columns) == {'kind', 'x', 'power'})
threshold = power.loc[power['kind'].eq('threshold')]
require('single cached H0 threshold', len(threshold) == 1 and finite(threshold['power']))
for kind in ['vol', 'drift']:
    curve = power.loc[power['kind'].eq(kind)].sort_values('x')
    require(f'{kind} power finite and bounded',
            finite(curve['power']) and curve['power'].between(0, 100).all())
    require(f'{kind} injected-shock grid monotone',
            (np.diff(curve['x'].to_numpy()) > 0).all())
    require(f'{kind} power non-decreasing',
            (np.diff(curve['power'].to_numpy()) >= 0).all())

roc = pd.read_parquet(ROC_PATH)
require('ROC schema', set(roc.columns) == {'grid', 'val', 'win', 'rho'})
require('ROC unique grid-window keys', ~roc.duplicated(['grid', 'val', 'win']).any())
require('ROC finite and non-negative', finite(roc['rho']) and roc['rho'].ge(0).all())
group_sizes = roc.groupby(['grid', 'val']).size()
require('ROC balanced 300-window groups', group_sizes.eq(300).all())
h0 = roc.loc[roc['grid'].eq('H0'), ['win', 'rho']].set_index('win')['rho']
auc_rows = []
for (grid, val), group in roc.loc[~roc['grid'].eq('H0')].groupby(['grid', 'val']):
    alt = group.set_index('win')['rho'].reindex(h0.index)
    auc = roc_auc_score(np.r_[np.zeros(len(h0)), np.ones(len(alt))],
                        np.r_[h0.values, alt.values])
    auc_rows.append((grid, float(val), float(auc)))
print(pd.DataFrame(auc_rows, columns=['shock', 'intensity', 'AUC']).to_string(index=False,
      float_format=lambda x: f'{x:.3f}'))

mc = pd.read_parquet(MC_PATH)
require('Monte-Carlo schema',
        set(mc.columns) == {'regime', 'window_end', 'L', 'mean', 'std', 'cv_pct'})
require('Monte-Carlo unique keys', ~mc.duplicated(['regime', 'L']).any())
require('Monte-Carlo finite positive diagnostics',
        finite(mc[['mean', 'std', 'cv_pct']])
        and mc[['mean', 'std', 'cv_pct']].gt(0).all().all())
expected_L = [10, 20, 50, 100, 200, 500, 1000]
expected_windows = {'Stress (GFC)': '2009-09', 'Calm (2013-2016)': '2016-06'}
require('Monte-Carlo exact regime set', set(mc['regime']) == set(expected_windows))
for regime, group in mc.groupby('regime'):
    group = group.sort_values('L')
    require(f'{regime}: exact L grid', group['L'].tolist() == expected_L)
    require(f'{regime}: expected window', group['window_end'].nunique() == 1
            and group['window_end'].iloc[0] == expected_windows[regime])
    require(f'{regime}: convergence endpoint', group['cv_pct'].iloc[-1] < group['cv_pct'].iloc[0])
    print(f'{regime}: CV {group["cv_pct"].iloc[0]:.2f}% -> {group["cv_pct"].iloc[-1]:.2f}%')

## 6. Robustness-series alignment

Construction variants are compared only on a common article sample. Correlations are descriptive outputs; the hard contract is exact temporal alignment and numerical validity.

In [ ]:
ref = article_signals['V_uni.parquet']
corr_rows = []
for fname in SIGNAL_FILES:
    if fname in {'V_uni.parquet', 'V_small_uni.parquet'}:
        continue
    series = article_signals[fname]
    require(f'{fname}: exact reference alignment', series.index.equals(ref.index))
    corr = float(ref.corr(series))
    require(f'{fname}: finite reference correlation', np.isfinite(corr))
    corr_rows.append((fname, corr))

corr_table = pd.DataFrame(corr_rows, columns=['variant', 'corr_with_reference'])
print(corr_table.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

## 7. Independent volatility-spanning reconstruction

Monthly realised volatility uses only the contemporaneous active PIT universe. Long-horizon realised volatility is rebuilt month by month from the exact 100 formation members and their complete 36-month daily window. This is the strict same-universe, same-window volatility control used to audit Section 5.

In [ ]:
sig_frame = signals['V_uni.parquet'][['date', 'rho']].copy()
sig_frame['period'] = sig_frame['date'].dt.to_period('M')
sig_sqrt = np.sqrt(sig_frame.set_index('period')['rho']).rename('sig')

vix = pd.read_parquet(VIX)[['date', 'vix']].copy()
vix['date'] = pd.to_datetime(vix['date'])
require('VIX unique dates', vix['date'].is_unique)
require('VIX finite and positive', finite(vix['vix']) and vix['vix'].gt(0).all())
iv_m = vix.set_index('date')['vix'].resample('ME').mean()
iv_m.index = iv_m.index.to_period('M')
iv_lh = iv_m.rolling(36, min_periods=36).mean()

active_returns = (
    pl.scan_parquet(PANEL)
    .filter(pl.col('is_active') == 1)
    .select(['DlyCalDt', 'DlyRet'])
    .collect(engine='streaming')
    .to_pandas()
)
active_returns['DlyCalDt'] = pd.to_datetime(active_returns['DlyCalDt'])
ew_active = active_returns.groupby('DlyCalDt')['DlyRet'].mean().sort_index()
rv_m = ew_active.resample('ME').std(ddof=1) * np.sqrt(252)
rv_m.index = rv_m.index.to_period('M')

rv_lh_values = {}
window_rows = []
for fm in article_fms:
    frame = formation_window(fm)
    ew_window = frame.mean(axis=1)
    period = pd.Period(fm, freq='M')
    rv_lh_values[period] = float(ew_window.std(ddof=1) * np.sqrt(252))
    window_rows.append((period, len(frame), frame.shape[1], frame.index.max()))
rv_lh = pd.Series(rv_lh_values, name='rv_lh').sort_index()

require('formation-window RV complete', rv_lh.index.equals(ARTICLE_MONTHS))
require('formation-window RV finite and positive', finite(rv_lh) and rv_lh.gt(0).all())
require('all reconstructed windows have 100 assets', all(r[2] == 100 for r in window_rows))

raw_vol = pd.concat({
    'sig': sig_sqrt, 'iv': iv_m, 'rv': rv_m,
    'iv_lh': iv_lh, 'rv_lh': rv_lh,
}, axis=1).loc[ARTICLE_START:ARTICLE_END]
df_m = raw_vol[['sig', 'iv', 'rv']].dropna()
df_lh = raw_vol[['sig', 'iv_lh', 'rv_lh']].dropna()
require('monthly spanning sample exact',
        df_m.index.equals(ARTICLE_MONTHS), f'n={len(df_m)}')
require('36-month spanning sample exact',
        df_lh.index.equals(ARTICLE_MONTHS), f'n={len(df_lh)}')

print(f'Monthly block: {df_m.index.min()} -> {df_m.index.max()} | n={len(df_m)}')
print(f'36-month block: {df_lh.index.min()} -> {df_lh.index.max()} | n={len(df_lh)}')

## 8. Persistence and regression reproducibility

Persistence tests are diagnostics, not binary evidence of stationarity. The twelve in-sample $R^2$ values are independently recomputed and compared with the frozen article snapshot using a small numerical tolerance. HAC affects inference, not $R^2$; joint models are nevertheless fitted with the declared 36-month Newey–West covariance for coefficient diagnostics.

In [ ]:
persistence_rows = []
for label, series in [
    ('sqrt_rho', raw_vol['sig']), ('IV_monthly', raw_vol['iv']),
    ('RV_monthly', raw_vol['rv']), ('IV_36m_MA', raw_vol['iv_lh']),
    ('RV_36m_formation', raw_vol['rv_lh']),
]:
    s = series.dropna()
    stat, pvalue, *_ = adfuller(s.values, autolag='AIC')
    persistence_rows.append((label, s.autocorr(1), stat, pvalue))
persistence = pd.DataFrame(persistence_rows, columns=['series', 'acf1', 'adf', 'pvalue'])
print(persistence.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

def fit(dep, regressors, frame):
    return sm.OLS(frame[dep], sm.add_constant(frame[regressors])).fit(
        cov_type='HAC', cov_kwds={'maxlags': HAC_LAGS})

def grid(frame, iv_key, rv_key):
    return {
        'IV': fit('sig', [iv_key], frame),
        'RV': fit('sig', [rv_key], frame),
        'both': fit('sig', [iv_key, rv_key], frame),
    }

dm_d, dl_d = df_m.diff().dropna(), df_lh.diff().dropna()
MODELS = {
    ('monthly', 'levels'): grid(df_m, 'iv', 'rv'),
    ('monthly', 'diffs'): grid(dm_d, 'iv', 'rv'),
    ('36-month', 'levels'): grid(df_lh, 'iv_lh', 'rv_lh'),
    ('36-month', 'diffs'): grid(dl_d, 'iv_lh', 'rv_lh'),
}

EXPECTED_R2 = {
    ('monthly', 'levels'): {
        'IV': 0.289236683,
        'RV': 0.132698246,
        'both': 0.346460797,
    },
    ('monthly', 'diffs'): {
        'IV': 0.100765162,
        'RV': 0.074320215,
        'both': 0.101928874,
    },
    ('36-month', 'levels'): {
        'IV': 0.727985496,
        'RV': 0.698636986,
        'both': 0.746795323,
    },
    ('36-month', 'diffs'): {
        'IV': 0.231968251,
        'RV': 0.569255977,
        'both': 0.573170064,
    },
}
R2_ATOL = 5e-4
r2_rows = []
for key, models in MODELS.items():
    for spec, model in models.items():
        actual = float(model.rsquared)
        expected = EXPECTED_R2[key][spec]
        require(f'R2 regression guard: {key[0]} {key[1]} {spec}',
                np.isclose(actual, expected, atol=R2_ATOL, rtol=0),
                f'actual={actual:.6f}, expected={expected:.6f}')
        r2_rows.append((*key, spec, actual))

r2_table = pd.DataFrame(r2_rows, columns=['horizon', 'transform', 'regressors', 'R2'])
print('\nIndependent R2 reconstruction:')
print(r2_table.pivot_table(index=['horizon', 'transform'], columns='regressors', values='R2')
      .to_string(float_format=lambda x: f'{x:.3f}'))

monthly_joint = MODELS[('monthly', 'levels')]['both']
long_joint = MODELS[('36-month', 'levels')]['both']
print('\nHAC(36) joint coefficient diagnostics:')
for label, model in [('monthly levels', monthly_joint), ('36-month levels', long_joint)]:
    vals = [(k, model.params[k], model.tvalues[k], model.pvalues[k])
            for k in model.params.index if k != 'const']
    print(label, ' | ', ' ; '.join(f'{k}={b:+.6f}, t={t:+.2f}, p={p:.3f}'
                                   for k, b, t, p in vals))

level_residual = 1 - MODELS[('36-month', 'levels')]['both'].rsquared
diff_residual  = 1 - MODELS[('36-month', 'diffs')]['both'].rsquared
print(f'\nScientific diagnostic: matched-horizon residual share '
      f'= {level_residual:.1%} in levels, {diff_residual:.1%} in differences.')

## 9. Article artifact manifest

The audit checks that every expected PDF and LaTeX table exists and is non-empty. It also rejects saved error outputs in the article notebook. Artifact content is not regenerated here.

In [ ]:
EXPECTED_FIGURES = [
    'R_01_signal_overview.pdf', 'R_02_event_study.pdf',
    'R_03_placebo_discrimination.pdf', 'R_04_power_curves.pdf',
    'R_05_roc_family.pdf', 'R_06_robust_M.pdf',
    'R_07_robust_lambda.pdf', 'R_08_robust_scaling.pdf',
    'R_09_robust_distance.pdf', 'R_10_mc_convergence.pdf',
    'R_11_robust_barycenter.pdf',
    'R_12a_robust_scaling_exponent_ts.pdf',
    'R_12b_robust_scaling_exponent.pdf',
    'R_13_vol_overlay.pdf',
]
EXPECTED_TABLES = [
    'R_13_placebo_discrimination.tex', 'R_14_power_curves.tex',
    'R_15_roc_family.tex', 'R_16_sensitivity_corr.tex',
    'R_17_vol_persistence.tex', 'R_18_vol_regression.tex',
]

for fname in EXPECTED_FIGURES:
    path = IMG / fname
    require(f'figure artifact: {fname}', path.exists() and path.stat().st_size > 1000)
    require(f'PDF signature: {fname}', path.read_bytes()[:4] == b'%PDF')

for fname in EXPECTED_TABLES:
    path = TAB / fname
    require(f'table artifact: {fname}', path.exists() and path.stat().st_size > 100)
    require(f'LaTeX tabular contract: {fname}', '\\begin{tabular}' in path.read_text())

article_nb = json.loads(NB_ART.read_text())
errors = []
for idx, cell in enumerate(article_nb.get('cells', [])):
    for output in cell.get('outputs', []):
        if output.get('output_type') == 'error':
            errors.append((idx, output.get('ename'), output.get('evalue')))
require('article notebook contains no saved error output', not errors, str(errors))
require('article notebook contains Section 5',
        any('## 5. Non-redundancy with volatility' in ''.join(c.get('source', []))
            for c in article_nb.get('cells', [])))

print(f'Artifacts checked: {len(EXPECTED_FIGURES)} figures, {len(EXPECTED_TABLES)} tables')

## 10. Conclusion

Reaching this cell means that all hard data, PIT, cache, regression-reproduction, and artifact contracts passed. Scientific diagnostics above must still be interpreted substantively; a passing test suite does not manufacture an economic conclusion.

In [ ]:
require('audit reached final cell', True)
print('\n' + '=' * 72)
print(f'ALL HARD CONTRACTS PASSED — {len(CHECKS)} checks')
print('Signal validation notebook is internally reproducible')
print('=' * 72)